# 6.4.3 Error Correction

:::{admonition} Prompts
:class: tip

- What is the no-cloning theorem? Why does it prevent classical repetition codes but not quantum error correction?
- Walk through the 3-qubit bit-flip code: how does syndrome measurement identify the error without collapsing the logical qubit?
- What is a stabilizer code? How do the stabilizer generators (parity checks) define the code space and enable error detection?
- How does the toric code (§2.3.3) implement quantum error correction topologically? What are the anyonic error syndromes?
- What does the threshold theorem guarantee, and why does it mean quantum computation is scalable in principle?
:::

## Lecture Notes

Decoherence destroys quantum information. But quantum mechanics itself provides the tools to fight back. By encoding a logical qubit in the entangled state of many physical qubits, errors can be detected and corrected — without ever measuring the logical state. This is the central miracle of quantum error correction, and it relies on a distinction that has no classical analog: measuring *what error occurred* without measuring *what state the qubit is in*.

### Overview

The no-cloning theorem rules out the simplest fix (backup copies), but entanglement enables something better: redundant encoding in a code subspace. Error syndromes — measured via stabilizer operators — reveal which physical qubit was corrupted, without disturbing the logical superposition. General stabilizer codes and topological codes (the toric code of §2.3.3) make this idea robust. The threshold theorem guarantees that below a critical error rate, arbitrarily reliable quantum computation is achievable by concatenating codes.

:::{admonition} Discussion
:class: dropdown tip

The threshold theorem says fault-tolerant quantum computation is possible if the physical error rate $p < p_\text{th}$. For the surface code, $p_\text{th} \approx 1\%$. How does the number of physical qubits per logical qubit scale with $p/p_\text{th}$? What happens as $p \to 0$?
:::

### No-Cloning Theorem

:::{admonition} No-Cloning Theorem
:class: important

There is no universal unitary operator that can copy an arbitrary unknown quantum state:

$$

U(\vert\psi\rangle \otimes \vert 0\rangle) = \vert\psi\rangle \otimes \vert\psi\rangle

$$

for all $\vert\psi\rangle$. This is a direct consequence of the linearity of quantum mechanics.
:::

:::{admonition} Proof: No-Cloning Theorem
:class: dropdown information

Suppose such a unitary $U$ exists. Test on orthogonal states:

$$

U(\vert 0\rangle \otimes \vert 0\rangle) = \vert 0\rangle \otimes \vert 0\rangle, \quad U(\vert 1\rangle \otimes \vert 0\rangle) = \vert 1\rangle \otimes \vert 1\rangle

$$

By linearity, acting on $\vert+\rangle = \frac{1}{\sqrt{2}}(\vert 0\rangle + \vert 1\rangle)$:

$$

U(\vert+\rangle \otimes \vert 0\rangle) = \frac{1}{\sqrt{2}}(\vert 0\rangle \otimes \vert 0\rangle + \vert 1\rangle \otimes \vert 1\rangle)

$$

But cloning would require:

$$

U(\vert+\rangle \otimes \vert 0\rangle) \stackrel{?}{=} \vert+\rangle \otimes \vert+\rangle = \frac{1}{2}(\vert 00\rangle + \vert 01\rangle + \vert 10\rangle + \vert 11\rangle)

$$

These are not equal (different entanglement structure). **Contradiction** — no such $U$ exists.
:::

**Consequence for error correction**: Classical repetition codes fail because you cannot clone $\vert\psi\rangle$. Quantum error correction must use a different strategy: distribute the *information* of $\vert\psi\rangle$ across multiple physical qubits without ever concentrating it in a single copyable form.

### Quantum Error Correction: 3-Qubit Bit-Flip Code

**Goal**: Protect one logical qubit from single-qubit bit-flip errors using 3 physical qubits.

:::{admonition} 3-Qubit Bit-Flip Code
:class: important

Encode the two logical basis states as:

$$

\vert 0\rangle_L = \vert 000\rangle, \quad \vert 1\rangle_L = \vert 111\rangle

$$ (eq-qec-3qubit-encoding)

A general logical state encodes as $\vert\psi\rangle_L = \alpha\vert 000\rangle + \beta\vert 111\rangle$. This is not a copy of $\vert\psi\rangle$ — the superposition is preserved but distributed.

**Key idea**: Measure *what error occurred* (syndrome), not *what state the qubit is in*.
:::

#### Error Scenarios

A single bit-flip ($\hat{\sigma}^x$ on qubit $j$) maps:
- Qubit 1: $\alpha\vert 000\rangle + \beta\vert 111\rangle \to \alpha\vert 100\rangle + \beta\vert 011\rangle$
- Qubit 2: $\to \alpha\vert 010\rangle + \beta\vert 101\rangle$
- Qubit 3: $\to \alpha\vert 001\rangle + \beta\vert 110\rangle$

Each corrupted state is orthogonal to all others — errors land in distinct subspaces.

#### Syndrome Measurement

Measure **parity checks** without measuring the logical data:

$$

\hat{P}_1 = \hat{Z}_1 \hat{Z}_2 \quad \text{(parity of qubits 1 and 2)}

$$ (eq-qec-parity-check-1)

$$

\hat{P}_2 = \hat{Z}_2 \hat{Z}_3 \quad \text{(parity of qubits 2 and 3)}

$$

| Error Location | $\hat{P}_1$ | $\hat{P}_2$ | Syndrome $(s_1, s_2)$ |
|---|---|---|---|
| None | $+1$ | $+1$ | $(0,0)$ |
| Qubit 1 | $-1$ | $+1$ | $(1,0)$ |
| Qubit 2 | $-1$ | $-1$ | $(1,1)$ |
| Qubit 3 | $+1$ | $-1$ | $(0,1)$ |

**Recovery**: Apply $\hat{\sigma}^x$ to the identified qubit. The logical state $\alpha\vert 000\rangle + \beta\vert 111\rangle$ is restored.

**Why syndrome measurement is non-destructive**: $\hat{P}_1$ and $\hat{P}_2$ measure parity (even/odd), not individual qubits. The superposition $\alpha\vert 000\rangle + \beta\vert 111\rangle$ is an eigenstate of $\hat{P}_1$ and $\hat{P}_2$ with eigenvalue $+1$ — measuring them reveals nothing about $\alpha$ or $\beta$.

:::{admonition} Derivation: Parity Checks Are Stabilizers of the Code Space
:class: dropdown information

We prove that measuring the syndrome operators $\hat{P}_1 = \hat{Z}_1\hat{Z}_2$ and $\hat{P}_2 = \hat{Z}_2\hat{Z}_3$ does not disturb any logical state. The key is that every codeword is a $+1$ eigenstate of both parity checks.

**Step 1: Verify code words are $+1$ eigenstates.**

For $\hat{P}_1 = \hat{Z}_1\hat{Z}_2$ (checks whether qubits 1 and 2 have the same value):

$$
\hat{P}_1\vert 000\rangle = (+1)(+1)\vert 000\rangle = +\vert 000\rangle
$$
$$
\hat{P}_1\vert 111\rangle = (-1)(-1)\vert 111\rangle = +\vert 111\rangle
$$

Similarly $\hat{P}_2\vert 000\rangle = +\vert 000\rangle$ and $\hat{P}_2\vert 111\rangle = +\vert 111\rangle$.

**Step 2: Superpositions are also $+1$ eigenstates.**

For any logical state $\vert\psi\rangle_L = \alpha\vert 000\rangle + \beta\vert 111\rangle$:

$$
\hat{P}_1\vert\psi\rangle_L = \alpha\hat{P}_1\vert 000\rangle + \beta\hat{P}_1\vert 111\rangle = \alpha\vert 000\rangle + \beta\vert 111\rangle = +\vert\psi\rangle_L
$$

So $\vert\psi\rangle_L$ is a $+1$ eigenstate of $\hat{P}_1$ regardless of $\alpha, \beta$.

**Step 3: Syndrome measurement is non-destructive.**

Since $\vert\psi\rangle_L$ is already an eigenstate of $\hat{P}_1$, measuring $\hat{P}_1$ yields outcome $+1$ with certainty and does not change the state (eigenstates are unchanged by measurements of their own observables). The same holds for $\hat{P}_2$.

**Step 4: Errors break the stabilizer condition.**

Apply $\hat{X}_1$ (bit-flip error on qubit 1): $\hat{X}_1\vert 000\rangle = \vert 100\rangle$, $\hat{X}_1\vert 111\rangle = \vert 011\rangle$. Now:

$$
\hat{P}_1(\alpha\vert 100\rangle + \beta\vert 011\rangle) = \alpha\hat{Z}_1\hat{Z}_2\vert 100\rangle + \beta\hat{Z}_1\hat{Z}_2\vert 011\rangle
$$
$$
= \alpha(-1)(+1)\vert 100\rangle + \beta(+1)(-1)\vert 011\rangle = -(\alpha\vert 100\rangle + \beta\vert 011\rangle)
$$

So the error state is a $-1$ eigenstate of $\hat{P}_1$ — the syndrome $s_1 = 1$ is detected without collapsing $\alpha$ and $\beta$. The syndrome tells us **which qubit was flipped** (the error location), not **what logical state is encoded** (the information). The two pieces of data are in orthogonal sectors of the Hilbert space.
:::

:::{admonition} Construction: Full 3-Qubit Code Details
:class: dropdown note

The encoding circuit uses two CNOT gates: $\vert\psi\rangle\vert 00\rangle \xrightarrow{\text{CNOT}_{12}} \xrightarrow{\text{CNOT}_{13}} \alpha\vert 000\rangle + \beta\vert 111\rangle$.

The two syndrome bits yield $2^2 = 4$ possible outcomes — enough to distinguish 3 errors + no error. This works because single-qubit errors land in orthogonal sectors of the 8-dimensional Hilbert space.

**Shor code extension**: The 9-qubit Shor code concatenates two 3-qubit codes to protect against both bit-flip and phase-flip errors on any single qubit, demonstrating that any single-qubit error can be corrected.
:::
:::{admonition} Derivation: Phase-Flip Code via Hadamard Conjugation and the Shor 9-Qubit Code
:class: dropdown information

The 3-qubit bit-flip code corrects $\hat{X}$ errors. An analogous construction corrects $\hat{Z}$ (phase-flip) errors, and **concatenating** the two gives the Shor code that corrects any single-qubit error.

**Step 1 — Hadamard conjugation maps $\hat{Z}$ to $\hat{X}$.** The Hadamard gate $H = (\hat{X}+\hat{Z})/\sqrt{2}$ satisfies $H\hat{Z}H = \hat{X}$ and $H\hat{X}H = \hat{Z}$. Therefore, a phase-flip $\hat{Z}_j$ in the computational basis becomes a bit-flip $\hat{X}_j$ in the Hadamard (X) basis. This is the key insight: work in the $\{\vert+\rangle, \vert-\rangle\}$ basis and phase flips become detectable bit flips.

**Step 2 — Phase-flip code codewords.** Encode:

$$
\vert 0\rangle_L = \vert{+}{+}{+}\rangle = \frac{1}{2\sqrt{2}}(\vert 0\rangle+\vert 1\rangle)^{\otimes 3}, \quad \vert 1\rangle_L = \vert{-}{-}{-}\rangle = \frac{1}{2\sqrt{2}}(\vert 0\rangle-\vert 1\rangle)^{\otimes 3}
$$

A phase-flip error $\hat{Z}_j$ maps $\vert+\rangle \to \vert-\rangle$, acting as a bit-flip in the X-basis. The syndrome operators are:

$$
\hat{X}_1\hat{X}_2, \quad \hat{X}_2\hat{X}_3
$$

(X-parity checks, analogous to $\hat{Z}_1\hat{Z}_2$ and $\hat{Z}_2\hat{Z}_3$ for the bit-flip code). Measuring these identifies which qubit was phase-flipped; applying $\hat{Z}$ to that qubit restores the codeword.

**Step 3 — Shor 9-qubit code by concatenation.** Encode each qubit of the phase-flip code using the bit-flip code. The logical codewords become:

$$
\vert 0\rangle_L = \frac{1}{2\sqrt{2}}(\vert 000\rangle+\vert 111\rangle)^{\otimes 3}, \quad \vert 1\rangle_L = \frac{1}{2\sqrt{2}}(\vert 000\rangle-\vert 111\rangle)^{\otimes 3}
$$

using 9 physical qubits. Each block of 3 is a bit-flip code protecting against $\hat{X}$; the three blocks together form a phase-flip code protecting against $\hat{Z}$.

**Step 4 — Correctability of any single-qubit error.** Any error on qubit $j$ decomposes as $E = \alpha I + \beta\hat{X}_j + \gamma\hat{Y}_j + \delta\hat{Z}_j$ with $\hat{Y}_j = \mathrm{i}\hat{X}_j\hat{Z}_j$. The syndrome detects:

| Error type | Detection mechanism |
|---|---|
| Bit flip $\hat{X}_j$ | $\hat{Z}_k\hat{Z}_l$ parity check within the 3-qubit block containing $j$ |
| Phase flip $\hat{Z}_j$ | $\hat{X}_k\hat{X}_l$ parity check between blocks |
| Both $\hat{Y}_j$ | Both sets of checks simultaneously |

The 8 syndrome bits distinguish all $3\times 4 = 12$ single-qubit errors (3 qubits per block, 4 error types) plus the no-error case. Since all error syndromes are distinct, the Knill-Laflamme conditions are satisfied for the full Pauli error set on any one qubit. The Shor code is an $[[9,1,3]]$ code. $\checkmark$
:::

### General Framework: Stabilizer Codes

:::{admonition} Stabilizer Code
:class: important

A stabilizer code is defined by a set of commuting Hermitian operators $\{\hat{S}_1, \hat{S}_2, \ldots, \hat{S}_k\}$ (stabilizers) with eigenvalues $\pm 1$. The **code space** is the common $+1$ eigenspace:

$$

\hat{S}_i\vert\psi\rangle_L = +\vert\psi\rangle_L \quad \forall\, i

$$

**Error detection**: An error $\hat{E}$ takes $\vert\psi\rangle_L$ out of the code space if $\hat{E}\hat{S}_i \neq \hat{S}_iE$ (anticommutes with some stabilizer). Measuring $\hat{S}_i$ after the error yields $-1$ — revealing the syndrome without disturbing the logical information.

**Code parameters**: An $[[n, k, d]]$ code uses $n$ physical qubits to encode $k$ logical qubits with distance $d$ (minimum weight of an undetectable logical error), correcting up to $\lfloor(d-1)/2\rfloor$ errors.
:::

:::{admonition} Knill-Laflamme Conditions
:class: important

A code with projector $\hat{\Pi}$ onto the code space can correct a set of errors $\{\hat{E}_a\}$ if and only if:

$$

\hat{\Pi} \hat{E}_a^\dagger \hat{E}_b \hat{\Pi} = C_{ab}\,\hat{\Pi}

$$ (eq-qec-knill-laflamme)

for all errors $\hat{E}_a, \hat{E}_b$, where $C_{ab}$ is a constant independent of the code state. This ensures that error information is encoded in the syndrome, not in the logical state.
:::


:::{admonition} Derivation: Why Knill-Laflamme Guarantees Correctability
:class: dropdown information

**Goal.** Show that $\hat{\Pi}\hat{E}_a^\dagger\hat{E}_b\hat{\Pi} = C_{ab}\hat{\Pi}$ is necessary and sufficient for the code to correct errors $\{\hat{E}_a\}$.

**Setup.** Let $\{\vert\bar{0}\rangle, \vert\bar{1}\rangle\}$ be orthonormal logical codewords spanning the code space (image of $\hat{\Pi}$). An arbitrary logical state is $\vert\bar{\psi}\rangle = \alpha\vert\bar{0}\rangle + \beta\vert\bar{1}\rangle$.

**Error occurs.** After error $\hat{E}_a$, the state is $\hat{E}_a\vert\bar{\psi}\rangle = \alpha\hat{E}_a\vert\bar{0}\rangle + \beta\hat{E}_a\vert\bar{1}\rangle$.

**Syndrome measurement.** For correction to work, the syndrome measurement must:
1. **Identify which error occurred** (distinguish $a$ from $b$).
2. **Not reveal any information about the logical state** (not distinguish $\alpha,\beta$).

**Condition analysis.** For two code states $\vert\bar{i}\rangle, \vert\bar{j}\rangle \in$ code space, the KL conditions require:

$$
\langle\bar{i}\vert\hat{E}_a^\dagger\hat{E}_b\vert\bar{j}\rangle = C_{ab}\,\delta_{ij}
$$

- **Off-diagonal ($i\neq j$)**: $\langle\bar{0}\vert\hat{E}_a^\dagger\hat{E}_b\vert\bar{1}\rangle = 0$. This ensures error $\hat{E}_a$ maps $\vert\bar{0}\rangle$ and $\vert\bar{1}\rangle$ to orthogonal error spaces — syndrome identifies the error without leaking logical info.

- **Diagonal ($i=j$)**: $\langle\bar{0}\vert\hat{E}_a^\dagger\hat{E}_b\vert\bar{0}\rangle = \langle\bar{1}\vert\hat{E}_a^\dagger\hat{E}_b\vert\bar{1}\rangle = C_{ab}$. This ensures the error syndrome value $C_{ab}$ is the same regardless of the logical state — measurement doesn't collapse the logical qubit.

**Recovery operation.** Given syndrome $a$ (error $\hat{E}_a$ occurred), apply $\hat{R}_a = \hat{E}_a^\dagger\hat{\Pi}$ (or more precisely the appropriate unitary within the error subspace). This maps:

$$
\hat{R}_a\hat{E}_a\vert\bar{\psi}\rangle = \hat{E}_a^\dagger\hat{E}_a\vert\bar{\psi}\rangle = C_{aa}\vert\bar{\psi}\rangle \propto \vert\bar{\psi}\rangle
$$

restoring the original logical state. The condition $C_{ab}$ being a $c$-number (not an operator) ensures recovery is a valid unitary on the code space. $\square$
:::

**Examples of stabilizer codes**:
- 3-qubit bit-flip: $[[3,1,1]]$; stabilizers $\{\hat{Z}_1\hat{Z}_2, \hat{Z}_2\hat{Z}_3\}$.
- Steane code: $[[7,1,3]]$; corrects any single-qubit error.
- Surface code: $[[n,1,\sqrt{n}]]$; local 2D interactions; leading hardware candidate.

### Connection to the Toric Code (§2.3.3)

The toric code — introduced in Ch2 as a topological model for anyons — is a quantum error correcting code in disguise. The stabilizers are the **vertex operators** $\hat{A}_v = \prod_{j\in v}\hat{\sigma}^x_j$ and **plaquette operators** $\hat{B}_p = \prod_{j\in p}\hat{\sigma}^z_j$ on a 2D lattice.

**Errors are anyons**: A bit-flip error $\hat{\sigma}^z$ on an edge anticommutes with the two vertex operators at its endpoints — creating a pair of anyonic $e$-excitations. The error syndrome is precisely the anyon locations, detectable by measuring $\hat{A}_v$.

**Topological protection**: Logical operators (undetectable errors) must traverse the entire torus. The code distance $d = L$ (system size), giving exponential suppression of logical errors with system size.

**Full circle**: The anyons of Ch2 are the error syndromes of Ch6. Topological quantum computing uses anyon braiding for fault-tolerant logic gates — connecting the course's two great themes: topology and quantum information.

### Threshold Theorem

If the physical error rate per gate $p$ is below a critical threshold $p_{\mathrm{th}}$, then by using sufficiently large quantum error correcting codes, the logical error rate can be made arbitrarily small. Quantum computation is **scalable in principle**.

:::{admonition} Threshold Theorem
:class: important

For a distance-$d$ code (such as the surface code), the logical error rate scales as

$$
p_{\mathrm{logical}} \sim \left(\frac{p}{p_{\mathrm{th}}}\right)^{(d+1)/2}
$$ (eq-qec-threshold-scaling)

If $p < p_{\mathrm{th}}$, increasing $d$ suppresses the logical error exponentially. If $p > p_{\mathrm{th}}$, increasing $d$ makes the logical qubit *worse*.
:::

For the surface code, $p_{\mathrm{th}} \approx 1\%$; current superconducting qubits achieve $p \sim 10^{-3}$, placing the ratio $p/p_{\mathrm{th}} \approx 0.1$.

:::{admonition} Derivation: Threshold Scaling Law
:class: dropdown information

The scaling {eq}`eq-qec-threshold-scaling` follows from a leading-order combinatorial argument.

**Step 1 — Minimum uncorrectable error weight.** A code with distance $d$ corrects any pattern of up to $t = \lfloor(d-1)/2\rfloor$ errors. A logical failure requires at least

$$
k_{\min} = \left\lceil\frac{d+1}{2}\right\rceil
$$

simultaneous physical errors. For the surface code, this corresponds to an error chain spanning more than halfway across the lattice.

**Step 2 — Failure probability.** Assuming independent errors with probability $p$, the logical error rate is

$$
p_{\mathrm{logical}} = \sum_{k=k_{\min}}^{n} N_k\, p^k (1-p)^{n-k},
$$

where $N_k$ counts the number of weight-$k$ error patterns that cause a logical failure.

**Step 3 — Small-$p$ approximation.** When $p \ll 1$, the sum is dominated by the lowest-weight term $k = k_{\min} = (d+1)/2$:

$$
p_{\mathrm{logical}} \approx N_{\min}\, p^{(d+1)/2},
$$

where $N_{\min}$ is the number of minimum-weight uncorrectable error paths (shortest chains across the lattice in the surface code).

**Step 4 — Normalize to the threshold.** The threshold $p_{\mathrm{th}}$ is defined as the physical error rate where increasing $d$ no longer suppresses $p_{\mathrm{logical}}$. Absorbing $N_{\min}$ into a normalized form gives the standard scaling law:

$$
p_{\mathrm{logical}} \sim \left(\frac{p}{p_{\mathrm{th}}}\right)^{(d+1)/2}.
$$

This is the standard scaling law for distance-$d$ topological codes. $\checkmark$
:::


**Why the threshold works**: Error correction introduces new errors (from imperfect syndrome measurements), but at rate $\sim p^2/p_{\mathrm{th}}$. When $p < p_{\mathrm{th}}$, error correction removes errors faster than it introduces them. Increasing the code distance $d$ suppresses logical errors exponentially in $d$.

### Summary

- The no-cloning theorem rules out backup copies of quantum states; quantum error correction encodes logical information in entangled states of many physical qubits instead.
- The 3-qubit bit-flip code detects and corrects single-qubit errors via syndrome measurement (parity checks $\hat{P}_1 = \hat{Z}_1\hat{Z}_2$, $\hat{P}_2 = \hat{Z}_2\hat{Z}_3$) without measuring the logical state.
- Stabilizer codes generalize this framework: stabilizers define the code space, syndrome measurements identify errors, and the Knill-Laflamme conditions characterize correctable errors.
- The toric code (§2.3.3) is a topological stabilizer code — its anyonic excitations are error syndromes, and topological protection gives distance $d = L$ (system size).
- The threshold theorem guarantees fault-tolerant quantum computation when $p < p_{\mathrm{th}} \approx 1\%$; for the surface code, $p_{\mathrm{logical}} \sim (p/p_{\mathrm{th}})^{(d+1)/2}$ — exponential suppression with code distance $d$.
- **Course comes full circle**: from qubits and their measurement (Ch1) to identical particles and anyons (Ch2) to the toric code protecting quantum information (Ch6). Quantum mechanics supports both the fragility of coherence and the tools to preserve it.

:::{admonition} See Also
:class: seealso

- **[2.3.3 Toric Code](../ch2_identical-particles/2-3-3-toric-code)**: Topological error correction and anyonic error syndromes
- **[6.4.1 Decoherence](6-4-1-decoherence)**: The decoherence that error correction must fight
- **[6.4.2 Lindblad Master Equation](6-4-2-lindblad-master-equation)**: Quantitative model of noise processes
:::

## Homework

**1.** Three-Qubit Bit-Flip Code: Encoding and Basis

The 3-qubit bit-flip code encodes logical qubits as:

$$

\vert 0\rangle_L = \vert 000\rangle, \quad \vert 1\rangle_L = \vert 111\rangle

$$

(a) Write the general logical state $\vert\psi\rangle_L = \alpha\vert 0\rangle_L + \beta\vert 1\rangle_L$ explicitly in terms of the physical qubits. Verify that this state lies in the 2-dimensional code subspace of $\mathbb{C}^8$.

(b) Show that a single bit-flip error $\hat{\sigma}^x_j$ on qubit $j$ maps both $\vert 0\rangle_L$ and $\vert 1\rangle_L$ to orthogonal states outside the code subspace.

(c) Explain why the syndrome measurement (parity checks $\hat{P}_1 = \hat{Z}_1 \hat{Z}_2$ and $\hat{P}_2 = \hat{Z}_2 \hat{Z}_3$) does *not* collapse the logical state but *does* reveal which qubit flipped.

**2.** Syndrome Measurement and Error Diagnosis

Consider the 3-qubit bit-flip code. Suppose the error channel acts on qubit 2, applying $\hat{\sigma}^x_2$ to the encoded state $\vert\psi\rangle_L = \alpha\vert 000\rangle + \beta\vert 111\rangle$.

(a) Write the corrupted state after the error occurs.

(b) Calculate the eigenvalues of the corrupted state under the two parity operators $\hat{P}_1 = \hat{Z}_1 \hat{Z}_2$ and $\hat{P}_2 = \hat{Z}_2 \hat{Z}_3$. Verify that the syndrome is $(s_1, s_2) = (1,1)$.

(c) Based on the syndrome table in the lecture notes, which qubit should be corrected? Apply the correction and verify that the original logical state is recovered.

**3.** Three-Qubit Phase-Flip Code

Now consider protection against phase-flip errors. Define the 3-qubit phase-flip code as:

$$

\vert 0\rangle_L = \vert+++\rangle = \frac{1}{2\sqrt{2}}(\vert 000\rangle + \vert 001\rangle + \vert 010\rangle + \vert 011\rangle + \vert 100\rangle + \vert 101\rangle + \vert 110\rangle + \vert 111\rangle)

$$

$$

\vert 1\rangle_L = \vert---\rangle = \frac{1}{2\sqrt{2}}(\vert 000\rangle - \vert 001\rangle - \vert 010\rangle + \vert 011\rangle - \vert 100\rangle + \vert 101\rangle + \vert 110\rangle - \vert 111\rangle)

$$

(a) Show that $\vert+\rangle_L$ and $\vert-\rangle_L$ are eigenstates of $\hat{Z}_1 \hat{Z}_2$ and $\hat{Z}_2 \hat{Z}_3$ with eigenvalue $+1$ and $-1$, respectively.

(b) Explain why a phase-flip error $\hat{\sigma}^z$ on any single qubit can be detected via the same parity checks.

(c) The phase-flip code can be understood as the bit-flip code applied in the Hadamard basis. How does the relationship $\hat{H} \hat{\sigma}^x \hat{H} = \hat{\sigma}^z$ connect the two codes?

**4.** Shor Code: Combining Bit-Flip and Phase-Flip Protection

The 9-qubit Shor code encodes one logical qubit into 9 physical qubits, protecting against both bit-flip and phase-flip errors on any single qubit. The encoding is:

$$

\vert 0\rangle_L = \frac{1}{2\sqrt{2}}(\vert 000\rangle + \vert 111\rangle) \otimes (\vert 000\rangle + \vert 111\rangle) \otimes (\vert 000\rangle + \vert 111\rangle)

$$

$$

\vert 1\rangle_L = \frac{1}{2\sqrt{2}}(\vert 000\rangle - \vert 111\rangle) \otimes (\vert 000\rangle - \vert 111\rangle) \otimes (\vert 000\rangle - \vert 111\rangle)

$$

Here each $(\vert 000\rangle + \vert 111\rangle)$ is a 3-qubit phase-flip protected block, and three such blocks protect against bit-flip errors.

(a) How many independent stabilizer generators (parity checks) are required to uniquely identify errors for the Shor code? Explain your reasoning.

(b) Write down *two* of the stabilizer generators explicitly, and explain what error type(s) each detects.

(c) Explain briefly how the Shor code achieves protection against both bit-flip and phase-flip errors with only 9 qubits. Why is this better than naively using two separate 3-qubit codes (which would require 6 qubits)?

**5.** Knill-Laflamme Conditions and Code Characterization

The **Knill-Laflamme conditions** state that a quantum error correction code can correct errors in set $\mathcal{E}$ if and only if:

$$

\langle \psi \vert \hat{E}_i^\dagger \hat{E}_j \vert \phi \rangle = \delta_{ij} f(i,j)

$$

for all $\vert\psi\rangle, \vert\phi\rangle$ in the code subspace and all error operators $\hat{E}_i, \hat{E}_j \in \mathcal{E}$, where $f(i,j)$ is independent of $\vert\psi\rangle$ and $\vert\phi\rangle$.

(a) For the 3-qubit bit-flip code, verify the Knill-Laflamme conditions for errors $E \in \{I, \hat{\sigma}^x_1, \hat{\sigma}^x_2, \hat{\sigma}^x_3\}$.

(b) Explain in physical terms why $\hat{E}_i^\dagger \hat{E}_j$ must satisfy this orthogonality structure for error correction to work.

(c) Why do the Knill-Laflamme conditions *not* allow the 3-qubit bit-flip code to correct two or more simultaneous bit-flip errors?

**6.** No-Cloning and Why Quantum Error Correction Is Non-Trivial

The no-cloning theorem states that an unknown quantum state cannot be universally copied. This creates a fundamental tension with error correction: how can you protect quantum information if you cannot make backup copies?

(a) Explain how the 3-qubit bit-flip code *avoids* cloning by distributing information across three physical qubits *without* copying the original state.

(b) Suppose you attempted to store a backup copy of a logical qubit by trying to create a second independent copy of the entire encoded state. Show that this would violate the no-cloning theorem.

(c) Quantum error correction requires measuring the error syndrome without measuring the logical data. Use the no-cloning theorem to argue why this is *necessary*: if you could measure logical data without destroying it, what would that imply about copying quantum states?

**7.** Threshold Theorem and Fault Tolerance (Conceptual)

The **threshold theorem** states: *If the physical error rate per qubit is below a critical threshold $p_{\text{th}}$, then quantum error correction codes can be concatenated recursively, with the logical error rate decreasing exponentially in the number of concatenation layers, enabling arbitrarily large quantum computations.*

(a) In simple terms, explain what "concatenation" of codes means and why it helps reduce errors.

(b) Current superconducting qubits have single-qubit gate errors around $10^{-3}$ to $10^{-4}$. Estimate what threshold $p_{\text{th}}$ would need to be achieved for practical fault-tolerant quantum computers. (Current estimates are $p_{\text{th}} \sim 10^{-3}$ to $10^{-4}$.)

(c) The threshold theorem shows that quantum computing can be **scalable in principle**. What are the remaining practical challenges—beyond just meeting the threshold—that would still stand in the way of a useful large-scale quantum computer?

(d) Surface codes are a leading candidate for fault-tolerant quantum computing. Without working through detailed calculations, explain why a 2D array of qubits (as in the surface code) might be more practical than fully connected qubits for building a large quantum computer, especially regarding error correction overhead.